# Phase 06A.01 — Blind taxonomy agreement, adjudication and schema-v2 freeze

Human annotations and provenance are mandatory. This notebook never synthesizes labels.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import save_json
from phase06a_common import EXPECTED_VALIDATION_ROWS, sha256_file
from phase06b_common import (MIN_TAXONOMY_KAPPA, build_taxonomy_manifest, taxonomy_agreement_report, validate_taxonomy_frame)
TD=ROOT/"outputs/phase06a/question_taxonomy"; IDS=ROOT/"data/splits/phase01/validation_sample_ids.json"
A1=TD/"annotator_1.csv"; A2=TD/"annotator_2.csv"; ADJ=TD/"taxonomy_adjudicated.csv"; FROZEN=TD/"taxonomy_frozen.csv"
PROV=TD/"taxonomy_human_provenance.json"; EXCEPTION=TD/"agreement_exception.json"
ids=json.loads(IDS.read_text()); assert len(ids)==EXPECTED_VALIDATION_ROWS


## Independent annotation and agreement gate

In [ ]:
required=[A1,A2]
if any(not p.is_file() for p in required):
    save_json(TD/"taxonomy_human_provenance.template.json",{"annotators":[{"name":"","completed_at_utc":""},{"name":"","completed_at_utc":""}],"adjudicator":{"name":"","completed_at_utc":""}})
    save_json(TD/"PHASE06A_01_STATUS.json",{"phase":"06A.01","status":"awaiting_annotation","required":[str(p) for p in required]})
    raise RuntimeError("Two independent human annotation files are required")
left=validate_taxonomy_frame(pd.read_csv(A1),ids); right=validate_taxonomy_frame(pd.read_csv(A2),ids)
report=taxonomy_agreement_report(left,right); disagreements=report.pop("disagreements")
save_json(TD/"agreement_metrics.json",report); disagreements.to_csv(TD/"adjudication_required.csv",index=False)
exception=json.loads(EXCEPTION.read_text()) if EXCEPTION.is_file() else None
exception_valid=isinstance(exception,dict) and all(str(exception.get(k,"")).strip() for k in ["approved_by","approved_at_utc","rationale"])
if not report["agreement_gate_passed"] and not exception_valid:
    save_json(TD/"PHASE06A_01_STATUS.json",{"phase":"06A.01","status":"awaiting_reannotation","failed_labels":report["failed_labels"],"threshold":MIN_TAXONOMY_KAPPA})
    raise RuntimeError("Kappa gate failed; recalibrate and re-annotate, or provide a signed exception")


## Human adjudication and immutable freeze

In [ ]:
missing=[p for p in [ADJ,PROV] if not p.is_file()]
if missing:
    save_json(TD/"PHASE06A_01_STATUS.json",{"phase":"06A.01","status":"awaiting_adjudication","missing":[str(p) for p in missing],"disagreement_rows":len(disagreements)})
    raise RuntimeError("Human adjudication and provenance are required")
frozen=validate_taxonomy_frame(pd.read_csv(ADJ),ids); frozen.to_csv(FROZEN,index=False)
provenance=json.loads(PROV.read_text())
manifest=build_taxonomy_manifest(taxonomy_path=FROZEN,validation_ids_path=IDS,annotator_1_path=A1,annotator_2_path=A2,adjudicated_path=ADJ,guideline_path=TD/"ANNOTATION_GUIDELINE.md",agreement=report,annotators=provenance.get("annotators"),adjudicator=provenance.get("adjudicator"),signed_exception=exception)
save_json(TD/"taxonomy_manifest.json",manifest)
save_json(TD/"PHASE06A_01_STATUS.json",{"phase":"06A.01","status":"complete","schema_version":2,"rows":len(frozen),"taxonomy_sha256":sha256_file(FROZEN)})
manifest


Agreement describes annotation reliability, not model performance. No freeze occurs below kappa 0.70 without signed provenance.